# LinkedIn Zip Solver

Solves LinkedIn Zip games in the browser. Reads the board directly from the page DOM, finds the path using DFS with pruning, and drags the mouse through the solution.

## Settings
Set guest mode (unlimited practice) or daily game, and whether to show the browser.

In [17]:
# ===========================================================================
# WINDOWS ASYNCIO EVENT LOOP FIX
# ===========================================================================
import sys
if sys.platform == "win32":
    import asyncio
    try:
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    except Exception:
        pass

# ===========================================================================
# GAME CONFIGURATION
# ===========================================================================
# False = Guest Mode (unlimited practice plays at linkedin.com/games/view/zip/desktop)
# True  = Logged-in Mode (solves your official daily LinkedIn puzzle)
LOGGED_IN: bool = True

# False = Opens a visible browser so you can watch the solver play live!
# True  = Runs completely silently in the background
HEADLESS: bool = False

# True  = Uses the deep learning model (zip_cnn.pth) as move prior
# False = Uses pure heuristic DFS (also extremely fast)
USE_CNN: bool = True

# Maximum DFS backtrack nodes before timing out
MAX_ITERATIONS: int = 5_000_000

mode_str = 'daily' if LOGGED_IN else 'guest'
disp_str = 'headless' if HEADLESS else 'window'
heur_str = 'with cnn' if USE_CNN else 'heuristic'
print(f'mode {mode_str} display {disp_str} solver {heur_str}')


mode daily display window solver with cnn


C:\Users\shiva\AppData\Local\Temp\ipykernel_26280\1279291560.py:8: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\shiva\AppData\Local\Temp\ipykernel_26280\1279291560.py:8: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


## Imports & Setup

In [18]:
import asyncio
import math
import os
import random
import re
import sys
import time
from collections import Counter
from dataclasses import dataclass, field
from enum import IntEnum
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import threading

import numpy as np

# Safe PyTorch import (graceful fallback if PyTorch is absent or DLL restricted)
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    HAS_TORCH = True
except (ImportError, OSError) as e:
    HAS_TORCH = False
    torch = None
    nn = None
    F = None
    print(f'pytorch not loaded {e} using heuristic mode')

# URLs
DEFAULT_GAME_URL: str = "https://www.linkedin.com/games/zip"
GUEST_GAME_URL: str = "https://www.linkedin.com/games/view/zip/desktop"
PERSISTENT_CONTEXT_DIR: Path = Path(".browser_data")
WEIGHTS_PATH: Path = Path("zip_cnn.pth")

class CellType(IntEnum):
    EMPTY: int = 0
    WALL: int = 1
    START: int = 2
    NUMBERED: int = 3
    GOAL: int = 4

MOVES: List[Tuple[int, int]] = [
    (-1, 0),  # 0 = UP
    (1, 0),   # 1 = DOWN
    (0, -1),  # 2 = LEFT
    (0, 1),   # 3 = RIGHT
]
MOVE_NAMES = ["UP", "DOWN", "LEFT", "RIGHT"]

@dataclass(slots=True)
class Cell:
    row: int
    col: int
    cell_type: int
    number: Optional[int] = None
    pixel_center: Tuple[int, int] = (0, 0)

@dataclass(slots=True)
class GameState:
    grid: np.ndarray
    numbers: np.ndarray
    start: Tuple[int, int]
    goal: Tuple[int, int]
    cells: List[Cell] = field(default_factory=list)
    canvas_bbox: Tuple[int, int, int, int] = (0, 0, 0, 0)
    cell_size: int = 40
    screenshot: Optional[np.ndarray] = None
    h_walls: Optional[np.ndarray] = None
    v_walls: Optional[np.ndarray] = None

    @property
    def shape(self) -> Tuple[int, int]:
        return self.grid.shape

    def pixel_of(self, row: int, col: int) -> Tuple[int, int]:
        x0, y0, _, _ = self.canvas_bbox
        cs = self.cell_size
        return (x0 + col * cs + cs // 2, y0 + row * cs + cs // 2)

print('dependencies loaded')


dependencies loaded


## Load CNN Model (Optional)
Loads zip_cnn.pth if present to help guide move ordering.

In [19]:
if HAS_TORCH:
    class ZipCNN(nn.Module):
        def __init__(self, in_channels: int = 4, num_actions: int = 4) -> None:
            super().__init__()
            self.conv1 = nn.Conv2d(in_channels, 32, 3, padding=1)
            self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
            self.conv3 = nn.Conv2d(64, 64, 3, padding=1)
            self.conv4 = nn.Conv2d(64, 64, 3, padding=1)
            self.bn1 = nn.BatchNorm2d(32)
            self.bn2 = nn.BatchNorm2d(64)
            self.bn3 = nn.BatchNorm2d(64)
            self.bn4 = nn.BatchNorm2d(64)

            self.policy_conv = nn.Conv2d(64, 2, 1)
            self.policy_fc = nn.Linear(2, 128)
            self.policy_head = nn.Linear(128, num_actions)

            self.value_conv = nn.Conv2d(64, 1, 1)
            self.value_fc = nn.Linear(1, 128)
            self.value_head = nn.Linear(128, 1)

        def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
            h = F.relu(self.bn1(self.conv1(x)))
            h = F.relu(self.bn2(self.conv2(h)))
            h2 = F.relu(self.bn3(self.conv3(h)))
            h = F.relu(self.bn4(self.conv4(h2))) + h2
            p_map = self.policy_conv(h)
            p = F.relu(self.policy_fc(p_map.mean(dim=(2, 3))))
            policy_logits = self.policy_head(p)
            v_map = self.value_conv(h)
            v = F.relu(self.value_fc(v_map.mean(dim=(2, 3))))
            value = torch.tanh(self.value_head(v))
            return policy_logits, value

        @torch.no_grad()
        def predict(self, state_tensor: torch.Tensor) -> Tuple[torch.Tensor, float]:
            if state_tensor.dim() == 3:
                state_tensor = state_tensor.unsqueeze(0)
            self.eval()
            logits, value = self.forward(state_tensor)
            policy_probs = F.softmax(logits, dim=1).squeeze(0)
            return policy_probs, float(value.item())
else:
    class ZipCNN:
        pass

def load_cnn_model(weights_path: Path = WEIGHTS_PATH, device: str = "cpu") -> Optional[ZipCNN]:
    if not HAS_TORCH:
        print('pytorch not found using heuristic mode')
        return None
    if not weights_path.exists():
        print('weights not found using heuristic mode')
        return None
    model = ZipCNN().to(device)
    try:
        state_dict = torch.load(weights_path, map_location=device)
        if isinstance(state_dict, dict) and "model" in state_dict:
            state_dict = state_dict["model"]
        model.load_state_dict(state_dict, strict=False)
        model.eval()
        print(f'loaded weights from {weights_path.name}')
        return model
    except Exception as exc:
        print(f'could not load weights {exc} using heuristic mode')
        return None

cnn_model = load_cnn_model() if (USE_CNN and HAS_TORCH) else None


loaded weights from zip_cnn.pth


## Solver Logic
DFS backtracker with deadlock checks, flood fill, and move ordering.

In [20]:
def can_move(grid: np.ndarray, h_walls, v_walls, r: int, c: int, nr: int, nc: int) -> bool:
    rows, cols = grid.shape
    if not (0 <= nr < rows and 0 <= nc < cols):
        return False
    if grid[nr, nc] == int(CellType.WALL):
        return False
    if h_walls is not None and v_walls is not None:
        if nr > r:    # Moving Down
            if h_walls[r, c]: return False
        elif nr < r:  # Moving Up
            if h_walls[nr, nc]: return False
        elif nc > c:  # Moving Right
            if v_walls[r, c]: return False
        elif nc < c:  # Moving Left
            if v_walls[nr, nc]: return False
    return True

def _unvisited_is_connected(grid: np.ndarray, visited: np.ndarray, start_r: int, start_c: int, h_walls=None, v_walls=None) -> bool:
    rows, cols = grid.shape
    unvisited = (grid != int(CellType.WALL)) & (~visited)
    total = int(unvisited.sum())
    if total == 0: return True

    seen = np.zeros_like(visited, dtype=bool)
    stack: List[Tuple[int, int]] = []
    for dr, dc in MOVES:
        nr, nc = start_r + dr, start_c + dc
        if 0 <= nr < rows and 0 <= nc < cols and unvisited[nr, nc]:
            stack.append((nr, nc))
            seen[nr, nc] = True
            break
    if not stack: return total == 0

    count = 0
    while stack:
        r, c = stack.pop()
        count += 1
        if count > total: break
        for dr, dc in MOVES:
            nr, nc = r + dr, c + dc
            if can_move(grid, h_walls, v_walls, r, c, nr, nc) and unvisited[nr, nc] and not seen[nr, nc]:
                seen[nr, nc] = True
                stack.append((nr, nc))
    return count == total

def _has_deadlock(grid: np.ndarray, visited: np.ndarray, cur_r: int, cur_c: int, h_walls=None, v_walls=None) -> bool:
    rows, cols = grid.shape
    for dr in range(-2, 3):
        for dc in range(-2, 3):
            r, c = cur_r + dr, cur_c + dc
            if not (0 <= r < rows and 0 <= c < cols):
                continue
            if grid[r, c] == int(CellType.WALL) or visited[r, c]:
                continue
            # Count available exits from cell (r, c)
            exits = 0
            for mdr, mdc in MOVES:
                nr, nc = r + mdr, c + mdc
                if can_move(grid, h_walls, v_walls, r, c, nr, nc):
                    if not visited[nr, nc] or (nr == cur_r and nc == cur_c):
                        exits += 1
            if exits == 0:
                return True  # Cell is completely trapped
    return False

class ZipSolver:
    def __init__(self, state: GameState, model: Optional[ZipCNN] = None, max_iterations: int = MAX_ITERATIONS, verbose: bool = True):
        self.state = state
        self.grid = state.grid
        self.rows, self.cols = state.shape
        self.model = model
        self.max_iterations = max_iterations
        self.verbose = verbose
        self.nodes_explored = 0
        self._total_cells = int((self.grid != int(CellType.WALL)).sum())

        numbered = [c for c in state.cells if c.cell_type in [int(CellType.START), int(CellType.NUMBERED), int(CellType.GOAL)]]
        numbered.sort(key=lambda c: c.number or 0)
        self.waypoints = [(c.row, c.col) for c in numbered]

    def _order_moves(self, r: int, c: int, visited: np.ndarray, next_wp_idx: int) -> List[int]:
        target = self.waypoints[next_wp_idx] if next_wp_idx < len(self.waypoints) else self.state.goal
        scores = []

        cnn_probs = None
        if self.model is not None and HAS_TORCH:
            t = torch.zeros(4, self.rows, self.cols, dtype=torch.float32)
            t[0] = torch.from_numpy((self.grid == int(CellType.WALL)).astype(np.float32))
            t[1, r, c] = 1.0
            t[2] = torch.from_numpy(visited.astype(np.float32))
            t[3, target[0], target[1]] = 1.0
            try:
                cnn_probs, _ = self.model.predict(t)
            except Exception:
                pass

        for idx, (dr, dc) in enumerate(MOVES):
            nr, nc = r + dr, c + dc
            if not can_move(self.grid, self.state.h_walls, self.state.v_walls, r, c, nr, nc) or visited[nr, nc]:
                continue
            # Warnsdorff score: count onward options
            onward = sum(1 for mdr, mdc in MOVES if can_move(self.grid, self.state.h_walls, self.state.v_walls, nr, nc, nr + mdr, nc + mdc) and not visited[nr + mdr, nc + mdc])
            dist = abs(nr - target[0]) + abs(nc - target[1])
            score = onward * 10.0 + dist
            if cnn_probs is not None:
                score -= float(cnn_probs[idx].item()) * 15.0
            scores.append((score, idx))

        scores.sort(key=lambda x: x[0])
        return [idx for _, idx in scores]

    def solve(self) -> Optional[List[Tuple[int, int]]]:
        sr, sc = self.state.start
        gr, gc = self.state.goal
        visited = np.zeros((self.rows, self.cols), dtype=bool)
        visited[sr, sc] = True
        path = [(sr, sc)]
        next_wp = 1 if len(self.waypoints) > 1 and self.waypoints[0] == (sr, sc) else 0

        t0 = time.perf_counter()
        ok = self._dfs(sr, sc, gr, gc, visited, next_wp, path)
        elapsed = time.perf_counter() - t0

        if self.verbose:
            print(f'solved in {elapsed:.3f}s exploring {self.nodes_explored} nodes')
        return path if ok else None

    def _dfs(self, r: int, c: int, gr: int, gc: int, visited: np.ndarray, next_wp: int, path: List[Tuple[int, int]]) -> bool:
        self.nodes_explored += 1
        if self.nodes_explored > self.max_iterations:
            return False

        if (r, c) == (gr, gc):
            return (int(visited.sum()) == self._total_cells and next_wp >= len(self.waypoints))

        for idx in self._order_moves(r, c, visited, next_wp):
            dr, dc = MOVES[idx]
            nr, nc = r + dr, c + dc

            ct = self.grid[nr, nc]
            if ct == int(CellType.NUMBERED):
                if next_wp >= len(self.waypoints) or (nr, nc) != self.waypoints[next_wp]:
                    continue
            if (nr, nc) == (gr, gc) and int(visited.sum()) + 1 < self._total_cells:
                continue

            visited[nr, nc] = True
            path.append((nr, nc))
            new_wp = next_wp + 1 if (next_wp < len(self.waypoints) and self.waypoints[next_wp] == (nr, nc)) else next_wp

            if not _unvisited_is_connected(self.grid, visited, nr, nc, self.state.h_walls, self.state.v_walls):
                visited[nr, nc] = False
                path.pop()
                continue

            if _has_deadlock(self.grid, visited, nr, nc, self.state.h_walls, self.state.v_walls):
                visited[nr, nc] = False
                path.pop()
                continue

            if self._dfs(nr, nc, gr, gc, visited, new_wp, path):
                return True

            visited[nr, nc] = False
            path.pop()
        return False


## Test on Sample Boards
Quick test on random boards before opening the browser.

In [21]:
def generate_synthetic_state(rows: int = 6, cols: int = 6, num_waypoints: int = 3, seed: Optional[int] = None) -> Tuple[GameState, List[Tuple[int, int]]]:
    rng = random.Random(seed)
    for _ in range(300):
        visited = np.zeros((rows, cols), dtype=bool)
        sr, sc = rng.randint(0, rows - 1), rng.randint(0, cols - 1)
        path = [(sr, sc)]
        visited[sr, sc] = True
        target_len = int(rows * cols * 0.7)
        while len(path) < target_len:
            r, c = path[-1]
            cand = [(r+dr, c+dc) for dr, dc in MOVES if 0 <= r+dr < rows and 0 <= c+dc < cols and not visited[r+dr, c+dc]]
            if not cand: break
            nr, nc = rng.choice(cand)
            path.append((nr, nc))
            visited[nr, nc] = True
        if len(path) < max(8, num_waypoints * 3): continue

        wp_count = min(num_waypoints, len(path) // 3)
        wp_indices = [0, len(path) - 1]
        if wp_count > 2:
            interior = list(range(2, len(path) - 2, max(1, len(path) // wp_count)))
            rng.shuffle(interior)
            wp_indices.extend(interior[:wp_count - 2])
        wp_indices = sorted(set(wp_indices))

        grid = np.full((rows, cols), int(CellType.WALL), dtype=np.int32)
        cells, numbered_cells = [], []
        path_set = set(path)
        for r in range(rows):
            for c in range(cols):
                if (r, c) in path_set:
                    idx = path.index((r, c))
                    if idx in wp_indices:
                        wp_num = wp_indices.index(idx) + 1
                        ct = int(CellType.START) if wp_num == 1 else (int(CellType.GOAL) if wp_num == len(wp_indices) else int(CellType.NUMBERED))
                        grid[r, c] = ct
                        cell = Cell(row=r, col=c, cell_type=ct, number=wp_num, pixel_center=(c * 40 + 20, r * 40 + 20))
                        cells.append(cell)
                        if ct == int(CellType.NUMBERED): numbered_cells.append(cell)
                    else:
                        grid[r, c] = int(CellType.EMPTY)
                        cells.append(Cell(row=r, col=c, cell_type=int(CellType.EMPTY), pixel_center=(c * 40 + 20, r * 40 + 20)))
                else:
                    cells.append(Cell(row=r, col=c, cell_type=int(CellType.WALL), pixel_center=(c * 40 + 20, r * 40 + 20)))

        numbers = np.array(sorted({c.number for c in numbered_cells if c.number is not None}), dtype=np.int32)
        state = GameState(grid=grid, numbers=numbers, start=path[0], goal=path[-1], cells=cells, canvas_bbox=(0, 0, cols * 40, rows * 40), cell_size=40)
        return state, path
    raise RuntimeError('Sanity board generation error')

print('testing solver on 5 sample boards')
all_passed = True
for seed in range(5):
    test_state, true_path = generate_synthetic_state(rows=6, cols=6, seed=seed)
    solver = ZipSolver(test_state, model=cnn_model, verbose=False)
    t0 = time.perf_counter()
    solved_path = solver.solve()
    elapsed = (time.perf_counter() - t0) * 1000
    ok = (solved_path is not None and solved_path[0] == test_state.start and solved_path[-1] == test_state.goal)
    all_passed = all_passed and ok
    status = '[PASS]' if ok else '[FAIL]'
    print(f'board {seed} solved in {elapsed:.1f}ms with {len(solved_path)} steps')

assert all_passed, 'Solver sanity check failed!'
print('all sample boards solved')


testing solver on 5 sample boards
board 0 solved in 21.4ms with 14 steps
board 1 solved in 15.2ms with 20 steps
board 2 solved in 12.5ms with 16 steps
board 3 solved in 18.1ms with 25 steps
board 4 solved in 13.9ms with 15 steps
all sample boards solved


## Browser DOM Extraction
Extracts grid dimensions, cell numbers, and wall positions directly from the page HTML.

In [22]:
_DOM_EXTRACTION_JS = '''
() => {
    const cells = document.querySelectorAll('[data-cell-idx]');
    if (cells.length === 0) return null;
    const result = { count: cells.length, cells: [], gridRect: null };
    cells.forEach(c => {
        const idx = parseInt(c.getAttribute('data-cell-idx'), 10);
        const rect = c.getBoundingClientRect();
        const content = c.querySelector('[data-cell-content="true"], .trail-cell-content');
        let number = null;
        if (content) {
            const aria = content.getAttribute('aria-label');
            if (aria) { const n = parseInt(aria, 10); if (!isNaN(n)) number = n; }
            else { const n = parseInt(content.textContent.trim(), 10); if (!isNaN(n)) number = n; }
        }
        result.cells.push({
            idx: idx, number: number, bg: window.getComputedStyle(c).backgroundColor,
            x: rect.x, y: rect.y, w: rect.width, h: rect.height
        });
    });
    const grid = document.querySelector('[data-testid="interactive-grid"]') || cells[0].parentElement;
    if (grid) {
        const gr = grid.getBoundingClientRect();
        result.gridRect = { x: gr.x, y: gr.y, w: gr.width, h: gr.height };
    }
    return result;
}
'''

def _parse_rgb(s: str) -> Optional[Tuple[int, int, int]]:
    m = re.match(r'rgba?\((\d+),\s*(\d+),\s*(\d+)', s.strip())
    return (int(m.group(1)), int(m.group(2)), int(m.group(3))) if m else None

def build_state_from_dom(dom_data: Dict[str, Any], screenshot: Optional[np.ndarray] = None) -> GameState:
    count = dom_data['count']
    side = int(round(math.sqrt(count)))
    rows, cols = side, side
    cells_data = dom_data['cells']

    # Wall classification via background color distance
    bgs = [_parse_rgb(cd['bg']) for cd in cells_data if _parse_rgb(cd['bg'])]
    active_mean = Counter(bgs).most_common(1)[0][0] if bgs else (255, 255, 255)

    wall_indices = set()
    for cd in cells_data:
        if cd['number'] is not None: continue
        rgb = _parse_rgb(cd['bg'])
        if rgb and math.sqrt(sum((a - b)**2 for a, b in zip(rgb, active_mean))) > 25.0:
            wall_indices.add(cd['idx'])

    gr = dom_data.get('gridRect') or {}
    gx, gy, gw, gh = gr.get('x', 0), gr.get('y', 0), gr.get('w', 0), gr.get('h', 0)
    cw, ch = gw / cols if cols else 40, gh / rows if rows else 40

    grid = np.zeros((rows, cols), dtype=np.int32)
    cells, numbered_cells = [], []

    for r in range(rows):
        for c in range(cols):
            idx = r * cols + c
            cd = cells_data[idx]
            num = cd['number']
            is_w = idx in wall_indices
            ct = int(CellType.NUMBERED) if (num is not None and not is_w) else (int(CellType.WALL) if is_w else int(CellType.EMPTY))
            px, py = int(round(gx + c * cw + cw / 2)), int(round(gy + r * ch + ch / 2))
            grid[r, c] = ct
            cell = Cell(row=r, col=c, cell_type=ct, number=num, pixel_center=(px, py))
            cells.append(cell)
            if ct == int(CellType.NUMBERED): numbered_cells.append(cell)

    numbered_cells.sort(key=lambda x: x.number or 0)
    start_cell, goal_cell = numbered_cells[0], numbered_cells[-1]
    grid[start_cell.row, start_cell.col] = int(CellType.START)
    grid[goal_cell.row, goal_cell.col] = int(CellType.GOAL)

    state = GameState(
        grid=grid, numbers=np.array([c.number for c in numbered_cells]),
        start=(start_cell.row, start_cell.col), goal=(goal_cell.row, goal_cell.col),
        cells=cells, canvas_bbox=(int(round(gx)), int(round(gy)), int(round(gw)), int(round(gh))),
        cell_size=int(round(min(cw, ch))), screenshot=screenshot,
    )
    return state


## Mouse Drag Automation
Simulates mouse dragging through the solution path in the browser.

In [23]:
def _cubic_bezier(p0, p1, p2, p3, t):
    u = 1.0 - t
    return (
        u**3 * p0[0] + 3*u**2*t * p1[0] + 3*u*t**2 * p2[0] + t**3 * p3[0],
        u**3 * p0[1] + 3*u**2*t * p1[1] + 3*u*t**2 * p2[1] + t**3 * p3[1],
    )

async def execute_drag_path(page, grid_path: List[Tuple[int, int]], state: GameState) -> bool:
    if not grid_path:
        print('path is empty')
        return False

    # Map grid coordinates to pixel centres
    cell_pixels = {(c.row, c.col): c.pixel_center for c in state.cells}
    pixel_points = [cell_pixels.get((r, c), state.pixel_of(r, c)) for r, c in grid_path]

    start_x, start_y = pixel_points[0]
    print(f'dragging mouse from {start_x} {start_y}')

    # Pre-drag hover & mouse down
    await page.mouse.move(start_x, start_y)
    await asyncio.sleep(0.05)
    await page.mouse.down()
    await asyncio.sleep(0.02)

    # Drag through each waypoint segment smoothly
    for i in range(len(pixel_points) - 1):
        p0 = pixel_points[i]
        p3 = pixel_points[i + 1]
        steps = 4
        for s in range(1, steps + 1):
            t = s / float(steps)
            curr_x = p0[0] + (p3[0] - p0[0]) * t
            curr_y = p0[1] + (p3[1] - p0[1]) * t
            await page.mouse.move(curr_x, curr_y)
            await asyncio.sleep(0.005)

    await page.mouse.up()
    await asyncio.sleep(0.2)
    print('mouse drag finished')
    return True


## Run Solver
Opens the browser, reads the board, finds the solution, and plays it.

In [24]:
import asyncio
import sys
import threading
from pathlib import Path

async def _solve_linkedin_zip_core():
    from playwright.async_api import async_playwright

    target_url = DEFAULT_GAME_URL if LOGGED_IN else GUEST_GAME_URL
    profile_dir = PERSISTENT_CONTEXT_DIR if LOGGED_IN else Path(".browser_guest")
    profile_dir.mkdir(parents=True, exist_ok=True)

    mode_str = 'logged in' if LOGGED_IN else 'guest'
    print(f'starting solver {mode_str}')

    pw = await async_playwright().start()
    try:
        context = await pw.chromium.launch_persistent_context(
            user_data_dir=str(profile_dir),
            headless=HEADLESS,
            viewport={"width": 1280, "height": 900},
            args=["--disable-blink-features=AutomationControlled"],
        )
    except Exception:
        context = await pw.chromium.launch_persistent_context(
            user_data_dir=str(profile_dir),
            headless=HEADLESS,
            viewport={"width": 1280, "height": 900},
        )

    page = context.pages[0] if context.pages else await context.new_page()
    print('opening page')
    await page.goto(target_url, timeout=45000)

    # Wait for the board or start button to load
    print('waiting for board')
    for _ in range(30):
        start_btn = await page.query_selector('button:has-text("Start"), button:has-text("Play"), button:has-text("Practice")')
        if start_btn and await start_btn.is_visible():
            print('clicking start')
            await start_btn.click()
            await asyncio.sleep(1.0)
            break
        dom_check = await page.evaluate(_DOM_EXTRACTION_JS)
        if dom_check and dom_check.get("count", 0) > 0:
            break
        await asyncio.sleep(1.0)

    # Extract DOM
    dom_data = await page.evaluate(_DOM_EXTRACTION_JS)
    if not dom_data or dom_data.get("count", 0) == 0:
        print('no active board found')
        await context.close()
        await pw.stop()
        return

    state = build_state_from_dom(dom_data)
    print(f'found {dom_data["count"]} cells {state.shape[0]}x{state.shape[1]}')

    # Solve
    print('finding path')
    solver = ZipSolver(state, model=cnn_model, max_iterations=MAX_ITERATIONS, verbose=True)
    solution_path = solver.solve()

    if not solution_path:
        print('no path found')
        await context.close()
        await pw.stop()
        return

    print(f'path found with {len(solution_path)} steps')
    print('dragging path on board')
    await execute_drag_path(page, solution_path, state)

    print('done')
    if not HEADLESS:
        for _ in range(60):
            if page.is_closed(): break
            await asyncio.sleep(1)

    await context.close()
    await pw.stop()

def _run_in_proactor_thread(coro_fn):
    res, err = None, None
    def _worker():
        nonlocal res, err
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            res = loop.run_until_complete(coro_fn())
        except Exception as e:
            err = e
        finally:
            loop.close()
    t = threading.Thread(target=_worker)
    t.start()
    t.join()
    if err:
        raise err
    return res

async def solve_linkedin_zip():
    """
    Main entrypoint for running the solver.
    Automatically detects if running in a Windows Jupyter kernel with SelectorEventLoop
    and delegates to a Proactor worker thread to avoid NotImplementedError.
    """
    try:
        current_loop = asyncio.get_running_loop()
    except RuntimeError:
        current_loop = None

    is_selector = sys.platform == "win32" and (
        current_loop is None or not isinstance(current_loop, getattr(asyncio, "ProactorEventLoop", ()))
    )

    if is_selector:
        return await asyncio.to_thread(_run_in_proactor_thread, _solve_linkedin_zip_core)
    else:
        return await _solve_linkedin_zip_core()

# Run solver directly (works with or without await)
await solve_linkedin_zip()


starting solver logged in
opening page
waiting for board
found 64 cells 8x8
finding path
solved in 9.467s exploring 9715 nodes
path found with 64 steps
dragging path on board
dragging mouse from 502 227
mouse drag finished
done
